 # 04 · Feature Engineering Trust & Safety — YelpConnect

 **Tipo:** Feature Engineering  
 **Dominio:** Trust & Safety  
 **Capa física:** Silver  
 **Origen:** entidades Silver conformadas  
 **Salida:** `workspace.yelp_silver.slv_review_features`

 ---

 ## Objetivo

 Construir características reutilizables a nivel Review para
 identificar posteriormente patrones potencialmente sospechosos
 mediante reglas heurísticas de Trust & Safety.

 ## Principios

 Este notebook:

 - construye variables derivadas;
 - conserva el grano de una Review;
 - genera evidencia reutilizable para reglas posteriores;
 - conserva trazabilidad hacia la ejecución Silver;
 - publica metadata funcional en Unity Catalog.

 Este notebook **NO**:

 - determina fraude;
 - asigna un nivel de riesgo;
 - genera candidatos de investigación;
 - ejecuta las reglas Trust & Safety;
 - elimina Reviews por comportamiento sospechoso.

 Las features son variables observables o derivadas.
 Su existencia por sí sola no representa fraude confirmado.

 ---

 ## Grano

 **1 fila = 1 Review**

 ---

 ## Familias de features

 ### Temporal

 - actividad del usuario en 30 minutos;
 - actividad del usuario en 1 hora;
 - actividad del usuario en 24 horas;
 - fecha, hora y día de semana.

 ### Usuario

 - antigüedad de cuenta;
 - cantidad de Reviews observadas.

 ### User → Business

 - Reviews del usuario hacia un Business;
 - concentración de actividad hacia el mismo Business.

 ### Texto

 - texto normalizado;
 - hash SHA-256;
 - longitud;
 - número de repeticiones.

 ### Rating

 - desviación del rating respecto al rating agregado del Business.

 ### Business

 - Reviews por día;
 - promedio diario;
 - desviación estándar;
 - spike ratio;
 - z-score diario.


In [0]:
# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import uuid

from datetime import (
    datetime,
    timezone
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"

SILVER = (
    f"{CATALOG}."
    f"{SILVER_SCHEMA}"
)

REVIEW_TABLE = (
    f"{SILVER}.slv_review"
)

USER_TABLE = (
    f"{SILVER}.slv_user"
)

BUSINESS_TABLE = (
    f"{SILVER}.slv_business"
)

FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

PIPELINE_NAME = (
    "E7_04_feature_engineering_ts"
)

FEATURE_EXECUTION_ID = str(
    uuid.uuid4()
)

EXECUTION_STARTED_AT = (
    datetime.now(
        timezone.utc
    )
)


print("=" * 72)
print(
    "YELPCONNECT - FEATURE ENGINEERING TRUST & SAFETY"
)
print("=" * 72)

print(
    f"Pipeline      : "
    f"{PIPELINE_NAME}"
)

print(
    f"Execution ID  : "
    f"{FEATURE_EXECUTION_ID}"
)

print(
    f"Inicio UTC    : "
    f"{EXECUTION_STARTED_AT.isoformat()}"
)

print(
    f"Review        : "
    f"{REVIEW_TABLE}"
)

print(
    f"User          : "
    f"{USER_TABLE}"
)

print(
    f"Business      : "
    f"{BUSINESS_TABLE}"
)

print(
    f"Output        : "
    f"{FEATURE_TABLE}"
)

print("=" * 72)


 
 # 1 · Precondiciones

 Feature Engineering depende de la capa Silver conformada
 y validada previamente.

 Se requieren:

 - `slv_review`;
 - `slv_user`;
 - `slv_business`.

 Además de su existencia, se valida un contrato mínimo de
 columnas para evitar errores tardíos o cambios silenciosos
 en las fuentes.

 El orden esperado del pipeline es:

 `Silver → DQ Silver → Feature Engineering T&S`


In [0]:
# COMMAND ----------

# ============================================================
# 3. VALIDAR SCHEMA SILVER
# ============================================================

try:

    spark.sql(
        f"""
        DESCRIBE SCHEMA
        {SILVER}
        """
    )

except Exception as exc:

    raise RuntimeError(
        f"""
        ❌ SCHEMA SILVER INEXISTENTE

        Schema:
        {SILVER}

        Ejecutar previamente:
        00_setup/01_configurar_lakehouse
        """
    ) from exc


print(
    f"✔ Schema disponible: "
    f"{SILVER}"
)

In [0]:

# COMMAND ----------

# ============================================================
# 4. VALIDAR TABLAS REQUERIDAS
# ============================================================

REQUIRED_TABLES = [
    REVIEW_TABLE,
    USER_TABLE,
    BUSINESS_TABLE
]


missing_tables = [

    table_name

    for table_name
    in REQUIRED_TABLES

    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_tables:

    raise RuntimeError(
        f"""
        ❌ TABLAS SILVER REQUERIDAS INEXISTENTES

        Tablas faltantes:

        {missing_tables}

        Ejecutar previamente:
        02_silver_transformacion
        y
        03_dq_silver
        """
    )


print(
    "✔ Tablas Silver disponibles."
)


In [0]:
# COMMAND ----------

# ============================================================
# 5. VALIDAR CONTRATOS DE ENTRADA
# ============================================================

REQUIRED_COLUMNS = {

    REVIEW_TABLE: {

        "review_id",
        "user_id",
        "business_id",

        "stars",
        "review_text",
        "review_ts",

        "useful",
        "funny",
        "cool",

        "_silver_execution_id"
    },

    USER_TABLE: {

        "user_id",
        "yelping_since"
    },

    BUSINESS_TABLE: {

        "business_id",
        "estrellas"
    }
}


for (
    table_name,
    expected_columns
) in REQUIRED_COLUMNS.items():

    actual_columns = set(
        spark.table(
            table_name
        ).columns
    )

    missing_columns = (
        expected_columns
        -
        actual_columns
    )

    if missing_columns:

        raise RuntimeError(
            f"""
            ❌ CONTRATO SILVER INCOMPLETO

            Tabla:
            {table_name}

            Columnas faltantes:
            {sorted(missing_columns)}
            """
        )


print(
    "✔ Contratos mínimos Silver validados."
)


# COMMAND ----------


 # 2 · Dataset base

 La construcción parte de `slv_review`.

 Se incorpora contexto mediante joins hacia:

 - `slv_user`;
 - `slv_business`.

 Los joins son `LEFT` porque Review representa la entidad
 principal del análisis.

 La ausencia de User debido a la cobertura parcial del dataset
 no debe provocar la eliminación de la Review ni interpretarse
 automáticamente como una señal Trust & Safety.


In [0]:

# COMMAND ----------

# ============================================================
# 6. REVIEWS
# ============================================================

reviews = (

    spark.table(
        REVIEW_TABLE
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "stars",
        "review_text",
        "review_ts",

        "useful",
        "funny",
        "cool",

        "_silver_execution_id"
    )
)


review_count = (
    reviews.count()
)


if review_count == 0:

    raise RuntimeError(
        """
        ❌ slv_review no contiene registros.

        Feature Engineering no puede continuar.
        """
    )


print(
    f"✔ Reviews Silver: "
    f"{review_count:,}"
)


In [0]:

# COMMAND ----------

# ============================================================
# 7. USERS
# ============================================================

users = (

    spark.table(
        USER_TABLE
    )

    .select(

        "user_id",

        "yelping_since"
    )
)

In [0]:
# COMMAND ----------

# ============================================================
# 8. BUSINESSES
# ============================================================

businesses = (

    spark.table(
        BUSINESS_TABLE
    )

    .select(

        "business_id",

        F.col(
            "estrellas"
        ).alias(
            "business_stars"
        )
    )
)


In [0]:
# COMMAND ----------

# ============================================================
# 9. BASE REVIEW + USER + BUSINESS
# ============================================================

features = (

    reviews

    .join(
        users,
        "user_id",
        "left"
    )

    .join(
        businesses,
        "business_id",
        "left"
    )
)


# COMMAND ----------

 # 3 · Features básicas

 En esta etapa se construyen variables básicas reutilizadas
 posteriormente por las diferentes familias de features.

 Se generan:

 - timestamp como Epoch;
 - fecha;
 - hora;
 - día de semana;
 - desviación del rating;
 - antigüedad de cuenta;
 - texto normalizado;
 - longitud de texto;
 - hash del texto normalizado.

 El hash se utiliza para comparar igualdad del texto normalizado
 sin utilizar el texto completo como clave de agrupación.

 Los textos vacíos no reciben hash para evitar agruparlos
 artificialmente como contenido repetido.

In [0]:
# COMMAND ----------

# ============================================================
# 10. FEATURES BÁSICAS
# ============================================================

features = (

    features

    # --------------------------------------------------------
    # Timestamp como Epoch
    # --------------------------------------------------------

    .withColumn(
        "review_epoch",

        F.col(
            "review_ts"
        ).cast(
            "long"
        )
    )

    # --------------------------------------------------------
    # Fecha
    # --------------------------------------------------------

    .withColumn(
        "review_date",

        F.to_date(
            "review_ts"
        )
    )

    # --------------------------------------------------------
    # Hora
    # --------------------------------------------------------

    .withColumn(
        "review_hour",

        F.hour(
            "review_ts"
        )
    )

    # --------------------------------------------------------
    # Día de semana
    #
    # Spark:
    # 1 = domingo
    # 7 = sábado
    # --------------------------------------------------------

    .withColumn(
        "review_day_of_week",

        F.dayofweek(
            "review_ts"
        )
    )

    # --------------------------------------------------------
    # Desviación del rating
    # Review vs Business
    # --------------------------------------------------------

    .withColumn(
        "rating_deviation",

        F.abs(
            F.col(
                "stars"
            )
            -
            F.col(
                "business_stars"
            )
        )
    )

    # --------------------------------------------------------
    # Antigüedad de la cuenta al momento de la Review
    # --------------------------------------------------------

    .withColumn(
        "account_age_days",

        F.datediff(

            F.to_date(
                "review_ts"
            ),

            F.to_date(
                "yelping_since"
            )
        )
    )

    # --------------------------------------------------------
    # Texto normalizado
    # --------------------------------------------------------

    .withColumn(
        "normalized_text",

        F.lower(

            F.regexp_replace(

                F.trim(
                    F.col(
                        "review_text"
                    )
                ),

                r"\s+",

                " "
            )
        )
    )

    # --------------------------------------------------------
    # Longitud del texto
    # --------------------------------------------------------

    .withColumn(
        "text_length",

        F.length(
            "normalized_text"
        )
    )

    # --------------------------------------------------------
    # Hash SHA-256 del texto normalizado
    #
    # Los textos vacíos o NULL no se agrupan.
    # --------------------------------------------------------

    .withColumn(
        "normalized_text_hash",

        F.when(

            F.length(
                F.col(
                    "normalized_text"
                )
            )
            >
            0,

            F.sha2(
                F.col(
                    "normalized_text"
                ),
                256
            )
        )
    )
)


# COMMAND ----------

 # 4 · Actividad temporal del usuario

 Se calcula cuántas Reviews ha publicado un usuario dentro de
 ventanas móviles anteriores a cada Review.

 Se utilizan:

 - 30 minutos;
 - 1 hora;
 - 24 horas.

 Cada ventana incluye la Review actual.

 Estas métricas representan **velocidad de actividad**.
 Una alta velocidad no implica automáticamente comportamiento
 fraudulento.


In [0]:
# COMMAND ----------

# ============================================================
# 11. WINDOWS TEMPORALES USER
# ============================================================

w_user_30m = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "review_epoch"
        )
    )

    .rangeBetween(
        -30 * 60,
        0
    )
)


w_user_1h = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "review_epoch"
        )
    )

    .rangeBetween(
        -60 * 60,
        0
    )
)


w_user_24h = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "review_epoch"
        )
    )

    .rangeBetween(
        -24 * 60 * 60,
        0
    )
)


In [0]:
# COMMAND ----------

# ============================================================
# 12. FEATURES TEMPORALES USER
# ============================================================

features = (

    features

    .withColumn(
        "user_reviews_30m",

        F.count(
            "*"
        ).over(
            w_user_30m
        )
    )

    .withColumn(
        "user_reviews_1h",

        F.count(
            "*"
        ).over(
            w_user_1h
        )
    )

    .withColumn(
        "user_reviews_24h",

        F.count(
            "*"
        ).over(
            w_user_24h
        )
    )
)


# COMMAND ----------


 # 5 · Comportamiento User y User → Business

 Se calculan:

 - cantidad total de Reviews observadas por usuario;
 - cantidad de Reviews del usuario hacia un mismo Business;
 - concentración relativa hacia ese Business.

 La concentración se define como:

 `user_business_review_count / user_review_count`

 Un valor elevado describe concentración de actividad,
 pero no constituye por sí mismo una determinación de riesgo.


In [0]:
# COMMAND ----------

# ============================================================
# 13. WINDOWS USER
# ============================================================

w_user = (

    Window

    .partitionBy(
        "user_id"
    )
)


w_user_business = (

    Window

    .partitionBy(
        "user_id",
        "business_id"
    )
)


In [0]:


# COMMAND ----------

# ============================================================
# 14. FEATURES USER / BUSINESS
# ============================================================

features = (

    features

    .withColumn(
        "user_review_count",

        F.count(
            "*"
        ).over(
            w_user
        )
    )

    .withColumn(
        "user_business_review_count",

        F.count(
            "*"
        ).over(
            w_user_business
        )
    )

    .withColumn(
        "user_business_concentration",

        F.when(

            F.col(
                "user_review_count"
            )
            >
            0,

            F.col(
                "user_business_review_count"
            )

            /

            F.col(
                "user_review_count"
            )
        )
    )
)


 # 6 · Repetición de texto

 Las Reviews se agrupan mediante el hash SHA-256 del
 texto normalizado.

 Para cada Review se calcula:

 `text_repeat_count`

 que indica cuántas Reviews comparten exactamente el mismo
 texto normalizado dentro del dataset disponible.

 Este valor constituye una característica de repetición,
 no una confirmación de contenido fraudulento.

In [0]:

# COMMAND ----------

# ============================================================
# 15. WINDOW TEXTO
# ============================================================

w_text = (

    Window

    .partitionBy(
        "normalized_text_hash"
    )
)


In [0]:
# COMMAND ----------

# ============================================================
# 16. TEXT REPEAT COUNT
# ============================================================

features = (

    features

    .withColumn(
        "text_repeat_count",

        F.when(

            F.col(
                "normalized_text_hash"
            ).isNotNull(),

            F.count(
                "*"
            ).over(
                w_text
            )
        )

        .otherwise(
            F.lit(
                0
            )
        )
    )
)

 # 7 · Actividad diaria del Business

 Para cada Business y fecha se calcula la cantidad de Reviews
 observadas.

 Posteriormente se calculan:

 - promedio diario;
 - desviación estándar diaria;
 - spike ratio;
 - z-score.

 ## Alcance del promedio

 El promedio considera únicamente días donde el Business tuvo
 Reviews dentro del dataset disponible.

 No se incorporan días sin actividad como valores cero.

 Esto debe considerarse al interpretar el `business_spike_ratio`.

In [0]:
# COMMAND ----------

# ============================================================
# 17. BUSINESS REVIEWS POR DÍA
# ============================================================

business_day = (

    features

    .groupBy(
        "business_id",
        "review_date"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "business_reviews_day"
        )
    )
)


In [0]:
# COMMAND ----------

# ============================================================
# 18. PROMEDIO Y DESVIACIÓN DIARIA BUSINESS
# ============================================================

business_average = (

    business_day

    .groupBy(
        "business_id"
    )

    .agg(

        F.avg(
            "business_reviews_day"
        ).alias(
            "business_avg_daily_reviews"
        ),

        F.stddev_pop(
            "business_reviews_day"
        ).alias(
            "business_std_daily_reviews"
        )
    )
)


In [0]:

# COMMAND ----------

# ============================================================
# 19. SPIKE FEATURES
# ============================================================

business_day = (

    business_day

    .join(
        business_average,
        "business_id",
        "left"
    )

    .withColumn(
        "business_spike_ratio",

        F.when(

            F.col(
                "business_avg_daily_reviews"
            )
            >
            0,

            F.col(
                "business_reviews_day"
            )

            /

            F.col(
                "business_avg_daily_reviews"
            )
        )
    )

    # --------------------------------------------------------
    # Z-score adicional
    #
    # Actualmente no es requerido por las reglas activas
    # R001-R006.
    #
    # Se conserva como feature auxiliar para evolución
    # posterior hacia detección estadística o ML.
    # --------------------------------------------------------

    .withColumn(
        "business_day_zscore",

        F.when(

            F.col(
                "business_std_daily_reviews"
            )
            >
            0,

            (
                F.col(
                    "business_reviews_day"
                )

                -

                F.col(
                    "business_avg_daily_reviews"
                )
            )

            /

            F.col(
                "business_std_daily_reviews"
            )
        )
    )
)


In [0]:

# COMMAND ----------

# ============================================================
# 20. JOIN BUSINESS DAILY FEATURES
# ============================================================

features = (

    features

    .join(

        business_day,

        [
            "business_id",
            "review_date"
        ],

        "left"
    )
)


# COMMAND ----------


 
 # 8 · Metadata de Feature Engineering

 Cada registro conserva metadata de la ejecución que generó
 las features.

 Se registran:

 - `feature_execution_id`;
 - `feature_processed_at`;
 - `feature_pipeline_name`;
 - `_silver_execution_id`.

 Esto permite identificar:

 - qué ejecución Silver originó la Review;
 - qué ejecución calculó las features;
 - cuándo fueron procesadas;
 - qué pipeline las generó.


In [0]:
# COMMAND ----------

# ============================================================
# 21. METADATA OPERACIONAL
# ============================================================

features = (

    features

    .withColumn(
        "feature_execution_id",

        F.lit(
            FEATURE_EXECUTION_ID
        )
    )

    .withColumn(
        "feature_processed_at",

        F.current_timestamp()
    )

    .withColumn(
        "feature_pipeline_name",

        F.lit(
            PIPELINE_NAME
        )
    )
)


# COMMAND ----------

 
 # 9 · Selección final

 La tabla final conserva únicamente atributos necesarios para
 el análisis Trust & Safety y su trazabilidad.

 El grano debe permanecer:

 **1 fila = 1 Review**

In [0]:

# COMMAND ----------

# ============================================================
# 22. SELECCIÓN FINAL
# ============================================================

features = (

    features

    .select(

        # ----------------------------------------------------
        # Keys
        # ----------------------------------------------------

        "review_id",
        "user_id",
        "business_id",

        # ----------------------------------------------------
        # Review
        # ----------------------------------------------------

        "stars",
        "review_text",
        "review_ts",

        "useful",
        "funny",
        "cool",

        # ----------------------------------------------------
        # Contexto
        # ----------------------------------------------------

        "business_stars",
        "yelping_since",

        # ----------------------------------------------------
        # Temporal
        # ----------------------------------------------------

        "review_date",
        "review_hour",
        "review_day_of_week",

        # ----------------------------------------------------
        # Rating
        # ----------------------------------------------------

        "rating_deviation",

        # ----------------------------------------------------
        # Account
        # ----------------------------------------------------

        "account_age_days",

        # ----------------------------------------------------
        # User velocity
        # ----------------------------------------------------

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",
        "user_review_count",

        # ----------------------------------------------------
        # User → Business
        # ----------------------------------------------------

        "user_business_review_count",
        "user_business_concentration",

        # ----------------------------------------------------
        # Texto
        # ----------------------------------------------------

        "normalized_text_hash",
        "text_length",
        "text_repeat_count",

        # ----------------------------------------------------
        # Business activity
        # ----------------------------------------------------

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_std_daily_reviews",
        "business_spike_ratio",
        "business_day_zscore",

        # ----------------------------------------------------
        # Trazabilidad
        # ----------------------------------------------------

        "_silver_execution_id",
        "feature_execution_id",
        "feature_processed_at",
        "feature_pipeline_name"
    )
)


# COMMAND ----------


 # 10 · Validación del grano

 Feature Engineering no debe multiplicar ni eliminar Reviews.

 Se validan dos condiciones:

 `features generadas = Reviews Silver`

 y:

 `review_id distintos = Reviews Silver`

 Si cualquiera de estas condiciones falla, el pipeline se detiene.

In [0]:
# COMMAND ----------

# ============================================================
# 23. VALIDAR GRANO
# ============================================================

feature_count = (
    features.count()
)


distinct_review_count = (

    features

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


print(
    f"Reviews Silver     : "
    f"{review_count:,}"
)

print(
    f"Features generadas : "
    f"{feature_count:,}"
)

print(
    f"Review ID únicos   : "
    f"{distinct_review_count:,}"
)


if (
    feature_count
    !=
    review_count

    or

    distinct_review_count
    !=
    review_count
):

    raise RuntimeError(
        f"""
        ❌ NO SE CUMPLE EL GRANO

        Grano esperado:
        1 Review = 1 fila

        Reviews Silver:
        {review_count}

        Features:
        {feature_count}

        Review IDs únicos:
        {distinct_review_count}
        """
    )


print(
    "✔ Grano 1 Review = 1 fila validado."
)


 
 # 11 · Persistencia

 `slv_review_features` se reconstruye completamente.

 La decisión es intencional porque varias features dependen del
 comportamiento global del dataset:

 - conteos por usuario;
 - concentración User → Business;
 - repetición de texto;
 - promedios diarios;
 - desviación estándar;
 - spikes.

 Un cálculo exclusivamente incremental podría generar resultados
 inconsistentes si no se recalcularan adecuadamente estas ventanas
 y agregaciones históricas.


In [0]:
# COMMAND ----------



# ============================================================
# 24. PERSISTENCIA
# ============================================================

(
    features

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        FEATURE_TABLE
    )
)


print(
    f"✔ Tabla publicada: "
    f"{FEATURE_TABLE}"
)


# COMMAND ----------

 # 12 · Gobierno en Unity Catalog

 `slv_review_features` es una tabla física Silver, pero su dominio
 funcional es **Trust & Safety**.

 Por esta razón se publica metadata específica en Unity Catalog.

 ## Clasificación

 La tabla se clasifica como `confidential` porque incluye:

 - `user_id`;
 - texto de Review;
 - variables de comportamiento;
 - características derivadas.

 ## Importante

 Los tags:

 - facilitan descubrimiento y gobierno;
 - documentan clasificación y propósito;
 - no constituyen por sí mismos controles de autorización.

 La seguridad mediante `GRANT`, vistas seguras u otras políticas
 se administra como una responsabilidad separada.

In [0]:

# COMMAND ----------

# ============================================================
# 25. METADATA UNITY CATALOG
# ============================================================

def apply_feature_uc_metadata(
    table_name
):

    description = (
        "Features derivadas a nivel Review para evaluación "
        "heurística Trust & Safety. Estas variables permiten "
        "identificar patrones potencialmente sospechosos, "
        "pero no representan fraude confirmado ni una decisión "
        "de investigación."
    )


    safe_description = (
        description.replace(
            "'",
            "''"
        )
    )


    # --------------------------------------------------------
    # COMMENT
    # --------------------------------------------------------

    spark.sql(
        f"""
        COMMENT ON TABLE {table_name}
        IS '{safe_description}'
        """
    )


    # --------------------------------------------------------
    # TAGS
    # --------------------------------------------------------

    try:

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TAGS (
                'data_layer' = 'silver',
                'data_domain' = 'trust_safety',
                'data_product' = 'ts_review_features',
                'source_system' = 'yelp',
                'classification' = 'confidential',
                'criticality' = 'high',
                'entity' = 'review_feature',
                'personal_data' = 'identifier'
            )
            """
        )

        print(
            f"✔ Tags UC aplicados: "
            f"{table_name}"
        )

    except Exception as exc:

        print(
            f"⚠ La tabla fue publicada, pero "
            f"no fue posible aplicar todos los tags UC."
        )

        print(
            f"Detalle: {exc}"
        )


apply_feature_uc_metadata(
    FEATURE_TABLE
)





 
 ## 12.1 · Diccionario funcional en Unity Catalog

 Las features que pueden resultar menos intuitivas reciben
 comentarios de columna.

 Esto permite comprender su significado directamente desde
 Unity Catalog sin necesidad de inspeccionar el código PySpark.


In [0]:

# COMMAND ----------

# ============================================================
# 26. COMMENTS DE COLUMNAS RELEVANTES
# ============================================================

COLUMN_COMMENTS = {

    "rating_deviation":
        (
            "Diferencia absoluta entre el rating de la Review "
            "y el rating agregado del Business."
        ),

    "account_age_days":
        (
            "Antigüedad estimada de la cuenta en días al "
            "momento de publicación de la Review."
        ),

    "user_reviews_30m":
        (
            "Cantidad de Reviews del usuario en una ventana "
            "móvil de 30 minutos, incluyendo la Review actual."
        ),

    "user_reviews_1h":
        (
            "Cantidad de Reviews del usuario en una ventana "
            "móvil de una hora, incluyendo la Review actual."
        ),

    "user_reviews_24h":
        (
            "Cantidad de Reviews del usuario en una ventana "
            "móvil de 24 horas, incluyendo la Review actual."
        ),

    "user_review_count":
        (
            "Cantidad total de Reviews del usuario observadas "
            "en el dataset Silver disponible."
        ),

    "user_business_review_count":
        (
            "Cantidad de Reviews del usuario asociadas al "
            "mismo Business."
        ),

    "user_business_concentration":
        (
            "Proporción de Reviews del usuario dirigidas al "
            "mismo Business dentro del dataset disponible."
        ),

    "normalized_text_hash":
        (
            "Hash SHA-256 del texto normalizado de la Review. "
            "Se utiliza para identificar igualdad textual."
        ),

    "text_length":
        (
            "Longitud del texto normalizado de la Review."
        ),

    "text_repeat_count":
        (
            "Cantidad de Reviews que comparten exactamente "
            "el mismo hash de texto normalizado."
        ),

    "business_reviews_day":
        (
            "Cantidad de Reviews observadas para un Business "
            "en la fecha de la Review."
        ),

    "business_avg_daily_reviews":
        (
            "Promedio de Reviews diarias del Business calculado "
            "sobre días con actividad observada."
        ),

    "business_std_daily_reviews":
        (
            "Desviación estándar poblacional de Reviews diarias "
            "del Business sobre días con actividad observada."
        ),

    "business_spike_ratio":
        (
            "Relación entre Reviews observadas en el día y el "
            "promedio diario del Business."
        ),

    "business_day_zscore":
        (
            "Z-score de la actividad diaria del Business respecto "
            "a su distribución histórica observada."
        ),

    "feature_execution_id":
        (
            "Identificador único de la ejecución que generó "
            "las features."
        ),

    "feature_processed_at":
        (
            "Timestamp de procesamiento de las features."
        ),

    "feature_pipeline_name":
        (
            "Nombre del pipeline responsable de construir "
            "las features."
        )
}


for (
    column_name,
    description
) in COLUMN_COMMENTS.items():

    safe_description = (
        description.replace(
            "'",
            "''"
        )
    )

    try:

        spark.sql(
            f"""
            ALTER TABLE {FEATURE_TABLE}
            ALTER COLUMN {column_name}
            COMMENT '{safe_description}'
            """
        )

    except Exception as exc:

        print(
            f"⚠ No fue posible aplicar COMMENT "
            f"a {column_name}: {exc}"
        )


print(
    "✔ Proceso de comentarios "
    "de columnas finalizado."
)


# COMMAND ----------


 # 13 · Reconciliación de persistencia

 La validación de grano anterior se realiza sobre el DataFrame.

 Después de escribir Delta se realiza una segunda reconciliación
 para verificar que la cantidad persistida coincida exactamente
 con las features generadas.

 Esta validación es técnica y no sustituye controles formales
 de Data Quality.


In [0]:

# COMMAND ----------

# ============================================================
# 27. VALIDAR PERSISTENCIA
# ============================================================

if not spark.catalog.tableExists(
    FEATURE_TABLE
):

    raise RuntimeError(
        f"""
        ❌ TABLA DE FEATURES NO PUBLICADA

        Tabla esperada:
        {FEATURE_TABLE}
        """
    )


persisted_feature_count = (

    spark.table(
        FEATURE_TABLE
    )

    .count()
)


if (
    persisted_feature_count
    !=
    feature_count
):

    raise RuntimeError(
        f"""
        ❌ RECONCILIACIÓN DE FEATURES FALLÓ

        Generadas:
        {feature_count}

        Persistidas:
        {persisted_feature_count}
        """
    )


print(
    f"✔ Features persistidas: "
    f"{persisted_feature_count:,}"
)


# COMMAND ----------

 # 14 · Evidencia

 Se presenta una muestra orientada a las principales familias
 de features.

 Esta visualización permite inspección técnica y no representa
 una lista de fraude ni una cola de investigación.

In [0]:


# ============================================================
# 28. EVIDENCIA
# ============================================================

display(

    spark.table(
        FEATURE_TABLE
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_business_concentration",

        "text_repeat_count",

        "business_reviews_day",
        "business_spike_ratio"
    )

    .orderBy(
        F.desc(
            "user_reviews_30m"
        )
    )

    .limit(
        100
    )
)


 # 15 · Perfil técnico rápido

 El perfil permite revisar magnitudes máximas de algunas features
 relevantes antes de ejecutar las reglas Trust & Safety.

 Los máximos se muestran como evidencia descriptiva.

 Un valor alto no implica automáticamente fraude.

In [0]:
# COMMAND ----------

# ============================================================
# 29. PERFIL RÁPIDO
# ============================================================

display(

    spark.table(
        FEATURE_TABLE
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "reviews"
        ),

        F.max(
            "user_reviews_30m"
        ).alias(
            "max_reviews_30m"
        ),

        F.max(
            "user_reviews_24h"
        ).alias(
            "max_reviews_24h"
        ),

        F.max(
            "text_repeat_count"
        ).alias(
            "max_text_repeat"
        ),

        F.max(
            "business_spike_ratio"
        ).alias(
            "max_business_spike_ratio"
        )
    )
)




 # Definition of Done — Feature Engineering T&S

 - [x] Schema Silver validado.
 - [x] Fuentes Silver verificadas.
 - [x] Contratos mínimos de entrada validados.
 - [x] Base Review + User + Business construida.
 - [x] Features temporales calculadas.
 - [x] Features de usuario calculadas.
 - [x] Features User → Business calculadas.
 - [x] Features de texto calculadas.
 - [x] Features de rating calculadas.
 - [x] Features de actividad Business calculadas.
 - [x] Metadata operacional agregada.
 - [x] Grano `1 Review = 1 fila` validado.
 - [x] `slv_review_features` publicada en Delta.
 - [x] Reconciliación de persistencia ejecutada.
 - [x] Metadata de Unity Catalog aplicada.
 - [x] Comentarios funcionales publicados.
 - [x] El notebook no asignó fraude.
 - [x] El notebook no asignó riesgo.
 - [x] El notebook no creó candidatos de investigación.

 ---

 ## Siguiente etapa

 Las features quedan disponibles para:

 `trust_safety_evaluacion`

 En la siguiente etapa se aplicarán las versiones activas
 del catálogo de reglas Trust & Safety.

 Las salidas deberán interpretarse como **señales heurísticas**
 y no como confirmación de fraude.


In [0]:

# COMMAND ----------

# ============================================================
# 30. RESULTADO FINAL
# ============================================================

EXECUTION_FINISHED_AT = (
    datetime.now(
        timezone.utc
    )
)


summary = {

    "status":
        "SUCCESS",

    "pipeline":
        PIPELINE_NAME,

    "execution_id":
        FEATURE_EXECUTION_ID,

    "started_at":
        EXECUTION_STARTED_AT
        .isoformat(),

    "finished_at":
        EXECUTION_FINISHED_AT
        .isoformat(),

    "grain":
        "1 review = 1 row",

    "source_reviews":
        review_count,

    "features_generated":
        feature_count,

    "features_persisted":
        persisted_feature_count,

    "output_table":
        FEATURE_TABLE,

    "data_domain":
        "trust_safety",

    "data_product":
        "ts_review_features"
}


print()
print("=" * 72)
print(
    "FEATURE ENGINEERING TRUST & SAFETY FINALIZADO"
)
print("=" * 72)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("=" * 72)


dbutils.notebook.exit(
    json.dumps(
        summary
    )
)

